# 01 Data Cleaning

## 1.1 Business Objective

This notebook prepares transaction-level e-commerce sales data for SKU-level demand analysis, replenishment planning, and warehouse allocation.

The final project aims to answer three business questions:

1. Which SKUs have stable demand and should be prioritized for local warehouse inventory?
2. Which SKUs show slow movement or high volatility and may create overstock risk?
3. How can sales data be transformed into a structured input for replenishment and warehouse decision support?

## 1.2 Data Source

This project uses the public UCI Online Retail dataset as a transaction-level sales data source. The dataset contains online retail transactions, including invoice number, product code, product description, quantity, invoice date, unit price, customer ID, and country.

Additional inventory and warehouse-related fields will be simulated in later steps for portfolio demonstration purposes. No confidential company data is used.

## 1.3 Cleaning Strategy

The raw dataset contains cancelled invoices, negative quantities, invalid prices, missing descriptions, duplicate rows, and non-product transaction lines.

For demand forecasting and inventory analysis, the main dataset should only include valid physical product sales transactions.

The cleaning process will:

1. Remove duplicate rows.
2. Separate cancellations and returns.
3. Remove invalid sales records with non-positive quantity or unit price.
4. Remove records without product descriptions.
5. Retain missing Customer ID records when they are still valid product sales transactions.
6. Exclude non-product transaction lines such as postage, fees, discounts, and manual adjustments.
7. Create a SKU master table using `stock_code` as the normalized SKU key.
8. Create revenue, invoice month, invoice week, and date fields.
9. Export cleaned transaction data and SKU-level outputs for downstream analysis.

## 1.4 Load Raw Data

The first step is to load the raw Excel file and inspect the transaction-level structure.

This confirms the number of rows, columns, and original fields before applying any cleaning logic.

In [27]:
import pandas as pd
from pathlib import Path

# Define file paths
raw_path = Path("../data/raw/Online Retail.xlsx")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Load raw Excel file
df = pd.read_excel(raw_path)

print("Raw data shape:", df.shape)
df.head()

Raw data shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 1.5 Initial Data Review

Before cleaning the data, I review missing values, duplicate rows, and data types.

This step is important because inventory and replenishment analysis depends on consistent transaction records. Incorrect quantities, cancelled invoices, invalid prices, or duplicate rows can distort SKU-level demand and lead to wrong replenishment recommendations.

In [28]:
# Standardize column names for easier analysis
df.columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country"
]

print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Column names:
['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']

Missing values:
invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

Duplicate rows:
5268

Data types:
invoice_no              object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[us]
unit_price             float64
customer_id            float64
country                    str
dtype: object


## 1.6 Key Data Issues Identified

The raw dataset includes several issues that need to be handled before demand analysis:

- Some invoices are cancellations, usually identified by invoice numbers starting with "C".
- Some records have negative quantities, which may represent returns or cancellations.
- Some records have zero or negative unit prices, which are not valid for sales revenue analysis.
- Some product descriptions are missing, making SKU classification difficult.
- Some rows are exact duplicates.
- Some stock codes represent non-product transaction lines such as postage, fees, discounts, or manual adjustments.

For this project, cancellations and returns are separated into a separate dataset. Non-product transaction lines are also separated from product sales data.

The main `clean_sales.csv` output will focus on valid physical product sales transactions.

## 1.7 Clean Valid Sales Transactions

The main sales dataset should represent valid product demand.

I remove exact duplicate rows, separate cancellations and returns, and keep only records with positive quantity, positive unit price, and valid product descriptions.

This creates the first version of the valid sales dataset before excluding non-product transaction lines.

In [29]:
# Remove exact duplicates
df_clean = df.drop_duplicates().copy()

# Convert invoice number to string
df_clean["invoice_no"] = df_clean["invoice_no"].astype(str)

# Separate cancellations and returns
returns_cancellations = df_clean[
    (df_clean["invoice_no"].str.startswith("C")) |
    (df_clean["quantity"] < 0)
].copy()

# Keep valid sales records only
sales = df_clean[
    (~df_clean["invoice_no"].str.startswith("C")) &
    (df_clean["quantity"] > 0) &
    (df_clean["unit_price"] > 0) &
    (df_clean["description"].notna())
].copy()

# Standardize text fields
sales["description"] = sales["description"].str.strip().str.upper()
sales["stock_code"] = sales["stock_code"].astype(str).str.strip()
sales["country"] = sales["country"].str.strip()

# Standardize date fields
sales["invoice_date"] = pd.to_datetime(sales["invoice_date"])
sales["invoice_date_only"] = sales["invoice_date"].dt.date
sales["invoice_month"] = sales["invoice_date"].dt.to_period("M").astype(str)
sales["invoice_week"] = sales["invoice_date"].dt.to_period("W").astype(str)

# Create business metric
sales["revenue"] = sales["quantity"] * sales["unit_price"]

print("Rows after removing duplicates:", len(df_clean))
print("Valid sales rows before excluding non-product lines:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))

sales.head()

Rows after removing duplicates: 536641
Valid sales rows before excluding non-product lines: 524878
Returns / cancellations rows: 10587


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_date_only,invoice_month,invoice_week,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,20.34


## 1.8 Treatment of Missing Customer ID

Customer ID is not required for this project's first stage because the analysis focuses on SKU-level demand, inventory planning, and warehouse allocation rather than customer segmentation.

Therefore, records with missing customer IDs are retained as long as they are valid sales transactions with positive quantity, positive unit price, valid product code, product description, and invoice date.

This avoids unnecessary loss of valid product sales records.

## 1.9 Excluding Non-Product Transaction Lines

The raw transaction data may include records that are not physical product sales, such as postage, fees, bank charges, discounts, or manual adjustments.

Since this project focuses on SKU-level demand forecasting, inventory planning, and warehouse allocation, these non-product transaction lines should not be included in the main product sales dataset.

These records are excluded from `clean_sales.csv` and saved separately as `non_product_lines.csv` for data transparency.

In [30]:
# Identify and exclude non-product transaction lines
non_product_stock_codes = [
    "POST",        # Postage
    "DOT",         # Dotcom postage / service-related line
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "D",           # Discount
    "M"            # Manual adjustment
]

non_product_lines = sales[
    sales["stock_code"].isin(non_product_stock_codes)
].copy()

sales = sales[
    ~sales["stock_code"].isin(non_product_stock_codes)
].copy()

print("Non-product transaction rows excluded:", len(non_product_lines))
print("Product sales rows after excluding non-product lines:", len(sales))

non_product_lines.head()

Non-product transaction rows excluded: 2162
Product sales rows after excluding non-product lines: 522716


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,invoice_date_only,invoice_month,invoice_week,revenue
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France,2010-12-01,2010-12,2010-11-29/2010-12-05,54.00
386,536403,POST,POSTAGE,1,2010-12-01 11:27:00,15.00,12791.0,Netherlands,2010-12-01,2010-12,2010-11-29/2010-12-05,15.00
1123,536527,POST,POSTAGE,1,2010-12-01 13:04:00,18.00,12662.0,Germany,2010-12-01,2010-12,2010-11-29/2010-12-05,18.00
1814,536544,DOT,DOTCOM POSTAGE,1,2010-12-01 14:32:00,569.77,NaN,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,569.77
2239,536569,M,MANUAL,1,2010-12-01 15:35:00,1.25,16274.0,United Kingdom,2010-12-01,2010-12,2010-11-29/2010-12-05,1.25


## 1.10 SKU Master Table

To create a more consistent SKU-level analysis, I create a SKU master table using `stock_code` as the normalized SKU key.

This avoids relying only on product descriptions, which may contain formatting differences or minor text variations.

The SKU master table summarizes each SKU's description, first sale date, last sale date, total units sold, total revenue, and average unit price.

In [31]:
sku_master = (
    sales
    .sort_values("invoice_date")
    .groupby("stock_code", as_index=False)
    .agg(
        description=("description", "first"),
        first_sale_date=("invoice_date", "min"),
        last_sale_date=("invoice_date", "max"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_unit_price=("unit_price", "mean")
    )
)

print("SKU master shape:", sku_master.shape)
sku_master.head()

SKU master shape: (3917, 7)


,stock_code,description,first_sale_date,last_sale_date,total_units,total_revenue,avg_unit_price
0,10002,INFLATABLE POLITICAL GLOBE,2010-12-01 08:45:00,2011-04-18 12:56:00,860,759.89,1.086620
1,10080,GROOVY CACTUS INFLATABLE,2011-02-27 13:47:00,2011-11-21 17:04:00,303,119.09,0.410909
2,10120,DOGGY RUBBER,2010-12-03 11:19:00,2011-12-04 13:15:00,192,40.32,0.210000
3,10123C,HEARTS WRAPPING TAPE,2010-12-03 11:19:00,2011-03-31 13:14:00,5,3.25,0.650000
4,10124A,SPOTS ON RED BOOKCOVER TAPE,2010-12-06 13:13:00,2011-11-06 13:00:00,16,6.72,0.420000


## 1.11 SKU Description Consistency Check

This step checks whether each `stock_code` maps to one or multiple product descriptions.

Since `stock_code` is used as the normalized SKU key, this check helps validate whether product descriptions are stable enough for reporting and analysis.

If one stock code has multiple descriptions, the SKU master table still uses `stock_code` as the primary key, while description is treated as a display field.

In [32]:
sku_description_check = (
    sales
    .groupby("stock_code", as_index=False)
    .agg(
        description_count=("description", "nunique"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum")
    )
    .sort_values("description_count", ascending=False)
)

sku_description_check.to_csv(processed_dir / "sku_description_check.csv", index=False)

sku_description_check.head(20)

,stock_code,description_count,total_units,total_revenue
2089,23236,4,2493,7059.97
2049,23196,4,1872,2842.24
2245,23413,3,173,909.31
2056,23203,3,20485,42030.34
2203,23366,3,2249,1667.95
2084,23231,3,7277,3008.34
104,17107D,3,170,433.50
2093,23240,3,4177,17000.17
1802,22937,3,972,2563.24
1980,23126,3,1109,5532.04


## 1.12 Monthly SKU-Level Sales Table

After removing cancellations, returns, invalid sales records, duplicate rows, and non-product transaction lines, the cleaned product sales data is aggregated to the monthly SKU level.

This table will be used in later notebooks for:

- SKU classification
- demand trend analysis
- seasonality review
- replenishment planning
- inventory risk identification
- warehouse allocation recommendations

This aggregation converts transaction-level sales records into a more useful time-series structure for product-level demand analysis.

In [33]:
monthly_sku_sales = (
    sales
    .groupby(["stock_code", "invoice_month"], as_index=False)
    .agg(
        monthly_units=("quantity", "sum"),
        monthly_revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
        avg_unit_price=("unit_price", "mean")
    )
)

monthly_sku_sales = monthly_sku_sales.merge(
    sku_master[["stock_code", "description"]],
    on="stock_code",
    how="left"
)

monthly_sku_sales = monthly_sku_sales[
    [
        "stock_code",
        "description",
        "invoice_month",
        "monthly_units",
        "monthly_revenue",
        "order_count",
        "avg_unit_price"
    ]
]

monthly_sku_sales.to_csv(processed_dir / "monthly_sku_sales.csv", index=False)

print("Monthly SKU sales shape:", monthly_sku_sales.shape)
monthly_sku_sales.head()

Monthly SKU sales shape: (34020, 7)


,stock_code,description,invoice_month,monthly_units,monthly_revenue,order_count,avg_unit_price
0,10002,INFLATABLE POLITICAL GLOBE,2010-12,251,234.41,30,1.201000
1,10002,INFLATABLE POLITICAL GLOBE,2011-01,340,291.37,21,0.962857
2,10002,INFLATABLE POLITICAL GLOBE,2011-02,52,45.76,7,1.072857
3,10002,INFLATABLE POLITICAL GLOBE,2011-03,28,27.70,8,1.142500
4,10002,INFLATABLE POLITICAL GLOBE,2011-04,189,160.65,5,0.850000


## 1.13 Save Cleaned Outputs

The cleaned outputs are saved for downstream notebooks.

At this stage, the project exports:

1. Valid product sales transactions.
2. Returns and cancellations.
3. Non-product transaction lines.
4. SKU master reference table.

In [34]:
# Save cleaned outputs
sales.to_csv(processed_dir / "clean_sales.csv", index=False)
returns_cancellations.to_csv(processed_dir / "returns_cancellations.csv", index=False)
non_product_lines.to_csv(processed_dir / "non_product_lines.csv", index=False)
sku_master.to_csv(processed_dir / "sku_master.csv", index=False)

print("Saved files:")
print("- data/processed/clean_sales.csv")
print("- data/processed/returns_cancellations.csv")
print("- data/processed/non_product_lines.csv")
print("- data/processed/sku_master.csv")

Saved files:
- data/processed/clean_sales.csv
- data/processed/returns_cancellations.csv
- data/processed/non_product_lines.csv
- data/processed/sku_master.csv


## 1.14 Data Quality Summary

This step creates a data quality summary table to document how the raw dataset was transformed into the final clean product sales dataset.

The purpose is to make the cleaning process transparent and reproducible.

In [35]:
data_quality_summary = pd.DataFrame({
    "step": [
        "raw_data",
        "after_duplicate_removal",
        "returns_cancellations_separated",
        "non_product_lines_excluded",
        "valid_product_sales_final",
        "monthly_sku_sales_records",
        "sku_master_records"
    ],
    "row_count": [
        len(df),
        len(df_clean),
        len(returns_cancellations),
        len(non_product_lines),
        len(sales),
        len(monthly_sku_sales),
        len(sku_master)
    ],
    "note": [
        "Original raw transaction rows",
        "Rows after removing exact duplicate records",
        "Cancellation and return-related records separated from main sales data",
        "Postage, fees, discounts, manual adjustments, and other non-product lines excluded",
        "Final valid physical product sales records",
        "Monthly SKU-level aggregation records",
        "Normalized SKU master records using stock_code as SKU key"
    ]
})

data_quality_summary.to_csv(processed_dir / "data_quality_summary.csv", index=False)

data_quality_summary

,step,row_count,note
0,raw_data,541909,Original raw transaction rows
1,after_duplicate_removal,536641,Rows after removing exact duplicate records
2,returns_cancellations_separated,10587,Cancellation and return-related records separa...
3,non_product_lines_excluded,2162,"Postage, fees, discounts, manual adjustments, ..."
4,valid_product_sales_final,522716,Final valid physical product sales records
5,monthly_sku_sales_records,34020,Monthly SKU-level aggregation records
6,sku_master_records,3917,Normalized SKU master records using stock_code...


## 1.15 Final Output Summary

This notebook produces seven processed outputs:

1. `clean_sales.csv`: valid product sales transactions used for demand and inventory analysis.
2. `returns_cancellations.csv`: cancellations and return-related records separated from the main sales dataset.
3. `non_product_lines.csv`: non-product transaction lines such as postage, fees, discounts, or manual adjustments.
4. `sku_master.csv`: normalized SKU-level reference table using `stock_code` as the main SKU key.
5. `sku_description_check.csv`: consistency check showing whether each `stock_code` maps to one or multiple product descriptions.
6. `monthly_sku_sales.csv`: monthly SKU-level sales aggregation used for downstream classification and replenishment analysis.
7. `data_quality_summary.csv`: summary table documenting the cleaning process and row-level data transformation.

These outputs create a clean and transparent data foundation for SQL business queries, SKU classification, and inventory decision support.

## 1.16 Management Implication

The cleaned product sales dataset, SKU master table, monthly SKU-level sales table, SKU description consistency check, and data quality summary create a reliable foundation for downstream business analysis.

With these outputs, management can later evaluate:

1. Which SKUs contribute the most sales volume and revenue.
2. Which SKUs show stable demand and should be prioritized for local inventory.
3. Which SKUs show low movement or high volatility and may require cautious replenishment.
4. Which SKUs may create stockout or overstock risk.
5. How historical sales can be converted into replenishment and warehouse allocation recommendations.

The next step is to use SQL to generate business summary tables, then classify SKUs and evaluate inventory risk.

In [36]:
print("===== 01 Data Cleaning Summary =====")
print("Valid product sales rows:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))
print("Non-product transaction rows excluded:", len(non_product_lines))
print("Monthly SKU sales shape:", monthly_sku_sales.shape)
print("SKU master shape:", sku_master.shape)

print("\nOutput files:")
print("- data/processed/clean_sales.csv")
print("- data/processed/returns_cancellations.csv")
print("- data/processed/non_product_lines.csv")
print("- data/processed/sku_master.csv")
print("- data/processed/monthly_sku_sales.csv")
print("- data/processed/data_quality_summary.csv")
print("- data/processed/sku_description_check.csv")

===== 01 Data Cleaning Summary =====
Valid product sales rows: 522716
Returns / cancellations rows: 10587
Non-product transaction rows excluded: 2162
Monthly SKU sales shape: (34020, 7)
SKU master shape: (3917, 7)

Output files:
- data/processed/clean_sales.csv
- data/processed/returns_cancellations.csv
- data/processed/non_product_lines.csv
- data/processed/sku_master.csv
- data/processed/monthly_sku_sales.csv
- data/processed/data_quality_summary.csv
- data/processed/sku_description_check.csv


## 1.17 Final Summary Code Cell

The final summary cell confirms the main output shapes and generated processed files from this notebook.